# Masterclass: LLM APIs — From 0 to Hero 🔌💻

Calling an LLM API isn't like querying a traditional relational database or calling a RESTful microservice.

**Traditional API:** GET /users/42 $\rightarrow$ Deterministic JSON payload. Same input, exact same output every time.

**LLM API:** POST /v1/chat/completions $\rightarrow$ Probabilistic text stream. It's expensive, asynchronous by nature, prone to rate limits, and returns unstructured text unless carefully constrained.

Let's examine all the nooks and corners of interacting with LLM APIs in production.

## Phase 1: Anatomy of an LLM API Request

Every major provider (OpenAI, Anthropic, Google Gemini, or local runners like Ollama) uses a variation of the JSON Chat Messages Payload.

A standard request payload looks like this:

In [ ]:
{
  "model": "gpt-5.5",
  "messages": [
    { "role": "system", "content": "You are a backend API migration expert. Always output responses in strict JSON format." },
    { "role": "user", "content": "Migrate this Express.js route to Fastify: app.get('/users', handler);" }
  ],
  "temperature": 0.1,
  "max_tokens": 500,
  "top_p": 0.95
}

### Understanding the Roles:

**system:** The overarching behavioral prompt or guardrail. It sets the persona, constraints, and output formats. (Crucial: Claude handles system prompts as a top-level parameter, whereas OpenAI includes it inside the messages array).

**user:** The actual query, document, or problem statement submitted by the end user.

**assistant:** The model's past replies. If you are building a multi-turn chat app, you must pass the historical array of user and assistant messages back and forth so the model maintains state (remember: LLMs are stateless).

## Phase 2: Streaming (Server-Sent Events)
Because LLMs generate text token by token sequentially, a large response can take 3 to 10 seconds to complete. If your web app waits for the entire response before rendering, your UI will feel sluggish and broken.

**The Solution — Server-Sent Events (SSE):**
When you set stream: true in your API request, the server opens an HTTP connection and pushes chunks down the wire as soon as each token is computed.

**Developer Implementation Pattern:**
In your frontend or backend router, you consume a readable stream, parsing incoming data chunks (data: {"choices": [{"delta": {"content": "ing"}}]}\n\n) and instantly appending them to the DOM or UI state. This gives users that familiar, real-time typing effect seen in ChatGPT.

## Phase 3: Essential Production Patterns & Safeguards

When writing code that hits LLM APIs, standard software engineering best practices apply tenfold.

### 1. Rate Limits and Token BucketsAPIs enforce strict limits measured in:
RPM (Requests Per Minute): How many times you can hit the endpoint.
TPM (Tokens Per Minute): The maximum volume of input + output tokens processed in 60 seconds.

The Fix: Implement robust exponential backoff retry logic using libraries like tenacity (Python) or async-retry (Node.js) to gracefully handle HTTP 429 Too Many Requests errors.

### 2. Cost Control & Hard BudgetsUnlike deterministic software where a loop costs negligible CPU cycles, every token costs real money.
Set hard monthly spending caps in your vendor dashboard (OpenAI/Anthropic console).
Use cheaper "Nano" or "Mini" models (e.g., GPT-4.1 Nano or GPT-5 Mini) for high-volume background tasks like text classification or intent routing, and reserve flagship models only for complex synthesis.

### 3. Structured Outputs (JSON Mode / Function Calling)

Developers used to struggle with LLMs adding conversational filler ("Sure, here is your JSON:") which broke downstream JSON parsers (JSON.parse()).

JSON Mode: Passing response_format: { "type": "json_object" } forces the model to guarantee syntactically valid JSON output.

Structured Outputs / Zod Integration: Modern APIs let you pass a JSON schema (or a Pydantic/Zod model structure), guaranteeing that the API response strictly matches your required data contract down to the field types.

### 4. Prompt Caching

If your system prompt or reference documents are large (e.g., a 50KB API documentation file passed with every user query), sending it raw every time balloons your costs.

Major APIs support Automatic Context Caching. By keeping your static system instructions and reference prefixes identical at the start of the payload, the provider caches those token embeddings in memory, offering up to a 75% to 90% discount on input token costs.  

## Phase 4: Developer Code Snippet (Python & Node.js Blueprint)



# Python (openai SDK)

In [ ]:
from openai import OpenAI

client = OpenAI()  # Automatically picks up OPENAI_API_KEY from environment

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "You are a helpful code assistant."},
        {"role": "user", "content": "Write a regex to validate an email address."},
    ],
    temperature=0.2,
    max_tokens=150,
)

print(response.choices[0].message.content)

# Node.js / TypeScript (@anthropic-ai/sdk)

In [ ]:
import Anthropic from '@anthropic-ai/sdk';

const anthropic = new Anthropic(); // Uses ANTHROPIC_API_KEY from environment

async function run() {
  const message = await anthropic.messages.create({
    model: 'claude-3-5-sonnet-latest',
    max_tokens: 300,
    temperature: 0.2,
    system: "You are a strict code refactoring tool.",
    messages: [{ role: 'user', content: 'Refactor this JavaScript loop to use map().' }]
  });

  console.log(message.content[0]);
}

run();